In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [35]:
# imports:
import torch
import torch.nn as nn
import transformers
from transformers import (
    TrainingArguments,
    Trainer,
    PretrainedConfig
)

In [36]:
# peft realted imports:
from peft import LoraConfig, get_peft_model
import tiktoken
import json
import random
import os
import sys
import gc

In [37]:
# saving paths:
proj_path = "/content/drive/MyDrive/llm_from_scratch/src"
data_path = "/content/drive/MyDrive/llm_from_scratch/datasets"
sys.path.append(proj_path)
sys.path.append(data_path)
os.chdir(proj_path)

In [38]:
# my local imports:
from gpt_model import GPTModel
from tiktoken_tokenizer import TiktokenTokenizer
# from config import GPTConfig

In [39]:
# selecting device:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [40]:
# from gpt_download import download_and_load_gpt2
# settings, params = download_and_load_gpt2(model_size="355M", models_dir="/content/drive/MyDrive/llm_from_scratch/src/gpt2")

In [41]:
# from load_weights import gpt as model

In [42]:
# torch.save(model.state_dict(), '/content/drive/MyDrive/llm_from_scratch/pretrained_weights.pth')

In [43]:
# import os

# file_path = '/content/drive/MyDrive/llm_from_scratch/pretrained_weights.pth'
# file_size = os.path.getsize(file_path)
# print(f"File size: {file_size} bytes")
# print(f"File size: {file_size / 1024 / 1024:.2f} MB")
# print(f"File size: {file_size / 1024 / 1024 / 1024:.2f} GB")

In [44]:
# from config import GPTConfig
# from gpt_model import GPTModel

# # Create model with SAME architecture as your weights file
# config = GPTConfig(
#     vocab_size=50257,
#     context_length=1024,
#     emb_dim=1024,  # Must match weights (1024 not 1280)
#     n_heads=16,    # Adjust based on your original model
#     n_layers=24,   # Must match weights (24+ layers)
#     drop_rate=0.1,
#     qkv_bias=True
# )

# model = GPTModel(config)
# model.load_state_dict(torch.load("/content/drive/MyDrive/llm_from_scratch/pretrained_weights.pth"))
# print("✅ Weights loaded successfully!")

In [45]:
# from config import GPTConfig
# from gpt_model import GPTModel
# CONFIG = GPTConfig()
# model =  GPTModel(CONFIG)

In [46]:
# # Save model with safe_serialization=False
# save_path = "/content/drive/MyDrive/llm_from_scratch/pretrained_gpt_model_and_tokenizer/my-gpt-model"

# model.save_pretrained(save_path, safe_serialization=False)
# print("✅ Model saved successfully!")

In [47]:
# from tiktoken_tokenizer import TiktokenTokenizer

# save_path = "/content/drive/MyDrive/llm_from_scratch/pretrained_gpt_model_and_tokenizer/my-gpt-model"

# # Save tokenizer
# tokenizer = TiktokenTokenizer(encoding_name="gpt2")
# tokenizer.save_pretrained(save_path)

# print("✅ Tokenizer saved!")

In [48]:
# import os

# save_path = "/content/drive/MyDrive/llm_from_scratch/pretrained_gpt_model_and_tokenizer/my-gpt-model"

# print("📁 Final File Structure:")
# files = os.listdir(save_path)
# for file in sorted(files):
#     file_path = os.path.join(save_path, file)
#     size_bytes = os.path.getsize(file_path)
#     size_mb = size_bytes / (1024*1024)

#     if size_mb < 0.1:  # Show KB for small files
#         print(f"   - {file} ({size_bytes/1024:.1f} KB)")
#     else:
#         print(f"   - {file} ({size_mb:.1f} MB)")

# print(f"\nTotal files: {len(files)}")

In [49]:
# fn to load dataset:
def load_and_convert_data(input_file):
    data = []
    with open(input_file, 'r') as f:
        for line in f:
            item = json.loads(line)
            if "conversations" in item and len(item["conversations"]) >= 2:
                data.append({
                    "messages": [
                        {"role": "user", "content": item["conversations"][0]},
                        {"role": "assistant", "content": item["conversations"][1]}
                    ]
                })
    return data

In [50]:
def load_alpaca_data(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

In [51]:

def load_orca_data(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)

    for item in data:
        if isinstance(item.get('messages'), str):
            item['messages'] = json.loads(item['messages'])
    return data

In [52]:
instruction_data = load_and_convert_data(f"{data_path}/instruction_data_lima.jsonl")

In [53]:
greeting_and_qa_data = json.load(open(f"{data_path}/greeting_and_qa.json"))

In [54]:
alpaca_data = load_alpaca_data(f"{data_path}/instruction_data_alpaca_gpt4.json")

In [55]:
orca_instruction_data = load_orca_data(f"{data_path}/orca_agentinstruct_10k_balanced.json")

In [56]:
# sampeling orca dataset for adding:
random.seed(44)
orca_instruction_data = random.sample(orca_instruction_data, 7418)
len(orca_instruction_data)

7418

## **Combining datasets to create a single one and shuffle it.**

In [57]:
merged_data = greeting_and_qa_data + orca_instruction_data + instruction_data + alpaca_data

In [58]:
random.seed(44)
random.shuffle(merged_data)

In [59]:
print(f"Total training examples: {len(merged_data)}")

Total training examples: 10000


In [60]:
class ConvertDataIntoTokenIds(torch.utils.data.Dataset):
    def __init__(self, data, max_length=512):
        self.data = data
        self.tokenizer = tiktoken.get_encoding('gpt2')
        self.max_length = max_length
        self.eot_token = 50256

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        user_message= assistant= ""

        if "instruction" in item:
            text = f"### Instruction:\n{item['instruction']}\n\n### Input:\n{item['input']}\n\n### Response:\n{item['output']}"

        elif item["messages"][0]["role"] == "system":
            user_msg = item["messages"][1]["content"]
            assistant_msg = item["messages"][2]["content"]
            text = f"User: {user_msg}\nAssistant: {assistant_msg}"

        else:
            user_msg = item["messages"][0]["content"]
            assistant_msg = item["messages"][1]["content"]
            text = f"User: {user_msg}\nAssistant: {assistant_msg}"

        tokens = self.tokenizer.encode(text)

        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]

        if len(tokens) < self.max_length:
            padding = [self.eot_token] * (self.max_length - len(tokens))
            tokens = tokens + padding

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'labels': torch.tensor(tokens, dtype=torch.long)
        }

In [61]:
# creating dataset:
eval_data = random.sample(merged_data, 244)
train_data = [item for item in merged_data if item not in eval_data]
train_dataset = ConvertDataIntoTokenIds(train_data)
eval_dataset = ConvertDataIntoTokenIds(eval_data)


In [62]:
print(len(train_dataset), len(eval_dataset))

9756 244


In [63]:
def load_custom_model():

    save_path = "/content/drive/MyDrive/llm_from_scratch/pretrained_gpt_model_and_tokenizer/my-gpt-model"

    print("Loading custom model...")
    model = GPTModel.from_pretrained(save_path)
    tokenizer = TiktokenTokenizer.from_pretrained(save_path)

    if torch.cuda.is_available():
        model = model.to("cuda")
        print("Model moved to GPU")

    print("Custom model loaded successfully!")
    return model, tokenizer

# model and tokenizer:
model, tokenizer = load_custom_model()


GPTModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Loading custom model...


GPTModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
GPTModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwar

Model moved to GPU
Custom model loaded successfully!


In [64]:
def setup_lora_model(model):

    target_modules = [
        "W_query", "W_key", "W_value",
        "out_proj",
        "out_head"
    ]

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=target_modules,
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_config)

    # print parameter counts:
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"LoRA applied successfully!")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Trainable percentage: {trainable_params/total_params*100:.2f}%")

    return model

# apply LoRA:
model = setup_lora_model(model)


LoRA applied successfully!
Total parameters: 356,806,280
Trainable parameters: 1,983,112
Trainable percentage: 0.56%


In [65]:
# training args:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/llm_from_scratch/finetuned_model/instruction_finetuned_model",

    # training hyperparameters
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=50,

    # mixed precision training for faster execution
    optim="adamw_torch",
    # optim="paged_adamw_8bit",
    fp16=True,

    # logging
    logging_steps=10,

    # evaluation settings
    eval_strategy="steps",
    eval_steps=50,
    prediction_loss_only=False,

    # output management
    overwrite_output_dir=True,
    remove_unused_columns=False
)

In [66]:
# trainer
import wandb
wandb.init(mode="disabled")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


In [67]:
print("Training started")

trainer.train()
print("Finetuning completed.")

Training started


Step,Training Loss,Validation Loss
50,14.359300,1.741521
100,14.341400,1.636930
150,14.642900,1.601097
200,13.919500,1.572565
250,13.184900,1.552877
300,13.166900,1.540761
350,13.442500,1.532078
400,13.363900,1.525344
450,13.435700,1.519768
500,13.985600,1.515342


Finetuning completed.


In [71]:
def test_model_fast(model, input_text="", max_new_tokens=512):
    encoding = tiktoken.get_encoding('gpt2')
    eot_token = encoding.eot_token

    # Tokenize
    input_ids = encoding.encode(input_text)
    input_tensor = torch.tensor([input_ids]).to(device)
    generated = input_tensor

    model.eval()
    with torch.no_grad():
        for i in range(max_new_tokens):
            # Forward pass - your model returns logits directly
            logits = model(generated)

            # Get last token logits and apply temperature
            next_token_logits = logits[0, -1, :] / 0.1

            # Convert to probabilities and sample
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            # Stop at EOT token
            if next_token.item() == eot_token:
                break

            # Append new token
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)

    # Return only the generated part (after input)
    full_output = encoding.decode(generated[0].tolist())
    return full_output[len(input_text):].replace('<|endoftext|>', '').strip()

In [72]:
# Test with your model
print("🧪 Testing the fine-tuned model:\n")
model = model.to(device)

test1 = test_model_fast(
    model,
    input_text="what is AI?",
)
print(f"\n{test1}\n")

🧪 Testing the fine-tuned model:



AcceleratorError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [79]:
torch.cuda.empty_cache()
gc.collect()

AcceleratorError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# # After instruction fine-tuning, merge LoRA into base model
# merged_model = model.merge_and_unload()

# # Save the MERGED weights (now base model has instruction knowledge)
# torch.save(merged_model.state_dict(), "/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model/finetuned_model.pth")

# print("✅ Saved unified model with instruction knowledge")